# The `lean_models` pipeline, interactively

This notebook walks the whole loop in one sitting: **write real Python in a
cell → extract it to an AST envelope → run it on the verified Lean
interpreter → differential-test it against CPython → prove a theorem about
it → watch a wrong spec fail** (and learn to read the failure).

**Before you start:** run `lake build` once at the repo root — the magics
reuse the build artifacts (`.olean` libraries and the `leanmodels-run`
binary). First run takes a few minutes; after that everything here is
seconds per cell.

The cell below locates the repo root and loads the magics from
`tools/lean_magic.py`. All scratch files live in `notebooks/work/`
(gitignored).

In [1]:
import pathlib, sys
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "lakefile.toml").exists())
sys.path.insert(0, str(root / "tools"))
%load_ext lean_magic

## Step 1 — write Python

`%%pyfile <name>.py` writes the cell body to `notebooks/work/<name>.py` and
runs the extractor (`extractors/python/extract.py`) on it. The extractor
never fails on valid Python: it emits the standardized JSON **envelope**
(`docs/envelope-schema.md`) in which anything outside the supported
vocabulary becomes an explicit `Unsupported` node. The summary below the
cell shows the functions found and flags those nodes.

In [2]:
%%pyfile square.py
def square(x):
    return x * x

The extractor wrote `notebooks/work/square.json` — the program's actual AST
as data — and the magic recompiled a persistent import header
(`NotebookHeader.olean`) so later `%%lean` cells can load every extracted
program from one precompiled module instead of re-parsing JSON per cell.

Here is what *honest partiality* looks like: true division needs floats,
which the v0 semantic tier doesn't have. The file still extracts — the `/`
node is **representable** but marked `Unsupported`, and the interpreter will
refuse to evaluate it loudly rather than guess.

In [3]:
%%pyfile pymath.py
def floordiv(a, b):
    return a // b

def mod(a, b):
    return a % b

def half(a):
    return a / 2        # true division -> floats -> outside the v0 tier

## Step 2 — run it on the verified interpreter

`%pyrun fn(args)` shells out to the `leanmodels-run` CLI: the same
fuel-based definitional interpreter your proofs will reason about, executed
on the envelope. It prints one line of canonical JSON; the magic
pretty-prints it. (Dotted form `%pyrun pymath.floordiv(-7, 2)` picks a file
explicitly; `--fuel N` overrides the default fuel of 10000.)

In [4]:
%pyrun square(12)

And the `Unsupported` node from above? *Loudly, never wrongly*: the run
reports `unsupported` instead of inventing a semantics.

In [5]:
%pyrun half(4)

## Step 3 — differential-test against CPython

Before proving anything about a function, this project's methodology demands
checking the executable model against ground truth — real CPython — not
against our own reading of the spec. `%pydiff fn args...` runs **both** and
compares canonical outcomes. A **MISMATCH is a failure**: the cell raises
after showing the side-by-side table, so a headless run
(`tools/run_notebooks.py`) catches a semantics regression instead of
printing it and moving on. (If a divergence is itself the lesson, tag the
cell `expected-error`.)

The interesting cases are where semantics *could plausibly diverge*.
Integer division on negatives is the classic: C, Rust, and Lean's own
`Int.div` **truncate** toward zero (`-7 / 2 = -3`), but Python **floors**
(`-7 // 2 = -4`). Likewise `%` follows the divisor's sign in Python
(`-7 % 3 = 2`, not `-1`). A careless "translate to Lean operators"
embedding would silently get these wrong. Watch them *not* diverge — the
Lean semantics uses `Int.fdiv`/`Int.fmod`, chosen because this very test
would fail otherwise:

In [6]:
%pydiff square 12
%pydiff floordiv -7 2
%pydiff mod -7 3

,outcome,canonical form
CPython (square.py),144,"{""status"": ""ok"", ""value"": {""t"": ""int"", ""v"": ""144""}}"
Lean interpreter,144,"{""status"": ""ok"", ""value"": {""t"": ""int"", ""v"": ""144""}}"


,outcome,canonical form
CPython (pymath.py),-4,"{""status"": ""ok"", ""value"": {""t"": ""int"", ""v"": ""-4""}}"
Lean interpreter,-4,"{""status"": ""ok"", ""value"": {""t"": ""int"", ""v"": ""-4""}}"


,outcome,canonical form
CPython (pymath.py),2,"{""status"": ""ok"", ""value"": {""t"": ""int"", ""v"": ""2""}}"
Lean interpreter,2,"{""status"": ""ok"", ""value"": {""t"": ""int"", ""v"": ""2""}}"


Exceptions are compared too — a raise is a first-class outcome, not a
harness failure:

In [7]:
%pydiff floordiv 1 0

,outcome,canonical form
CPython (pymath.py),raises ZeroDivisionError,"{""status"": ""exn"", ""exn"": ""ZeroDivisionError""}"
Lean interpreter,raises ZeroDivisionError,"{""status"": ""exn"", ""exn"": ""ZeroDivisionError""}"


## Step 4 — prove it

`%%lean` treats the cell as Lean code, checked against every program
extracted so far (via the precompiled header). Two things below:

* `#py_check square(12) = 144` — the **non-vacuity convention**: a concrete
  run checked at elaboration time. Theorems here are fuel-based partial
  correctness ("*if* the interpreter returns, then …"), so every example
  first demonstrates the "if" side is inhabited.
* `square(x) ==> x * x` — the total-correctness judgment: *the actual
  Python program* `square`, run through the verified interpreter on input
  `x`, terminates and returns `x * x`. No AST, no `Val`, no fuel in the
  statement (`docs/spec-surface.md`); `py_prove [square]` discharges it by
  symbolic execution.

In [8]:
%%lean square_proof
#py_check square(12) = 144

theorem square_spec (x : PyInt) : square(x) ==> x * x := by
  py_prove [square]

## Step 5 — what failure looks like

The goal state is the interface: when a spec is wrong, Lean shows you
exactly the mathematical claim that remains after symbolic execution. Here
is a deliberately wrong spec — `square(x) ==> x + x`. The cell errors (kept
in this notebook on purpose), and the goal reads:

```
x : PyInt
⊢ x * x = x + x
```

Symbolic execution already reduced *what the program computes* to `x * x`;
what's left is the (false) arithmetic claim against the spec's `x + x`.
When you see a stuck goal like this, the program side is settled — it's the
**spec** (or, for loops, the invariant — next notebook) that needs fixing.

In [9]:
%%lean square_wrong
theorem square_wrong (x : PyInt) : square(x) ==> x + x := by
  py_prove [square]

LeanCellError: Lean reported 1 error(s) in square_wrong — see the goal state above

## Where to next

* **`02-loops-interactive.ipynb`** — loops: discover an invariant by
  running the program, get stuck with the wrong one, read the residual
  goals, fix it.
* `docs/spec-surface.md` — the full judgment family (`==>`, `~~>`, `==>!`,
  `⇓`) and the spec-writing gallery.
* `Examples/python/<name>/` — the house style: pure `<name>.py` beside its
  generated envelope, `spec.lean` (statements) and `proof.lean` (proofs)
  — the three-file layout. `Examples/python/sum_to/sum_to.py` keeps the inline
  `# lean[` alternative shown above, end to end.